# Hampton Roads Flood Impact Reports

This notebook is the report runner for the final 7-city regional analysis. It refreshes the database-derived reports, loads the generated CSVs, and displays the tables most useful for the final writeup or presentation.

Frozen regional scope: Norfolk, Virginia Beach, Chesapeake, Hampton, Newport News, Portsmouth, and Suffolk.

## How To Use

Run the notebook top to bottom from the repository root. The heavy spatial workflow should already have been run. This notebook refreshes the final report CSVs from the database and displays them.

If you need to regenerate the city GeoPackages too, set `REFRESH_GIS_EXPORTS = True` in the setup cell.

## 1. Setup

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "data" / "processed" / "gis"
ACS_YEAR = 2023
TOP_ROAD_LIMIT = 10
REFRESH_GIS_EXPORTS = False

CITY_EXPORTS = {
    "norfolk_va": "norfolk_flood_exposure_1m.gpkg",
    "virginia_beach_va": "virginia_beach_flood_exposure_coarse.gpkg",
    "chesapeake_va": "chesapeake_flood_exposure_coarse.gpkg",
    "hampton_va": "hampton_flood_exposure_coarse.gpkg",
    "newport_news_va": "newport_news_flood_exposure_coarse.gpkg",
    "portsmouth_va": "portsmouth_flood_exposure_coarse.gpkg",
    "suffolk_va": "suffolk_flood_exposure_coarse.gpkg",
}

REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f"project_root={PROJECT_ROOT}")
print(f"report_dir={REPORT_DIR}")

In [ ]:
def run_command(args: list[str]) -> None:
    command = [str(part) for part in args]
    print("$", " ".join(command))
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    completed.check_returncode()

def money(value: float) -> str:
    return f"${value:,.0f}"

def load_report(name: str) -> pd.DataFrame:
    path = REPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing report: {path}")
    return pd.read_csv(path)

## 2. Refresh Report CSVs

These commands update ACS property-value exposure and regenerate all presentation-ready CSV outputs.

In [ ]:
run_command([sys.executable, "scripts/calculate_property_value_exposure.py", "--year", str(ACS_YEAR)])
run_command([
    sys.executable,
    "scripts/export_presentation_outputs.py",
    "--output-dir",
    str(REPORT_DIR),
    "--top-road-limit",
    str(TOP_ROAD_LIMIT),
])

In [ ]:
if REFRESH_GIS_EXPORTS:
    for study_area_id, filename in CITY_EXPORTS.items():
        run_command([
            sys.executable,
            "scripts/export_gis_layers.py",
            "--study-area-id",
            study_area_id,
            "--output",
            str(REPORT_DIR / filename),
        ])
else:
    print("Skipping GeoPackage refresh. Set REFRESH_GIS_EXPORTS = True to regenerate GIS exports.")

## 3. Load Reports

In [ ]:
regional = load_report("regional_flood_comparison.csv")
plus_3ft = load_report("regional_chart_plus_3ft_summary.csv")
metric_summary = load_report("regional_metric_summary.csv")
plus_3ft_metrics = load_report("regional_plus_3ft_metric_summary.csv")
top_roads = load_report("regional_top_impacted_roads.csv")
damage_chart = load_report("regional_chart_damage_by_city_scenario.csv")
buildings_chart = load_report("regional_chart_flooded_buildings_by_city_scenario.csv")
roads_chart = load_report("regional_chart_flooded_road_miles_by_city_scenario.csv")
property_chart = load_report("regional_chart_exposed_property_value_by_city_scenario.csv")

print(f"regional_rows={len(regional)}")
print(f"top_impacted_road_rows={len(top_roads)}")

## 4. Final +3 ft Results

In [ ]:
display_columns = [
    "study_area_name",
    "flooded_building_count",
    "flooded_road_miles",
    "estimated_damage_cost",
    "median_home_value",
    "estimated_exposed_property_value",
]
plus_3ft_display = plus_3ft[display_columns].copy()
plus_3ft_display["flooded_road_miles"] = plus_3ft_display["flooded_road_miles"].round(2)
for column in ["estimated_damage_cost", "median_home_value", "estimated_exposed_property_value"]:
    plus_3ft_display[column] = plus_3ft_display[column].map(money)
plus_3ft_display

## 5. Aggregate Metrics

In [ ]:
row = plus_3ft_metrics.iloc[0]
aggregate_rows = [
    {"metric": "Flooded buildings", "total": row.sum_flooded_building_count, "mean": row.mean_flooded_building_count, "median": row.median_flooded_building_count, "min": row.min_flooded_building_count, "max": row.max_flooded_building_count},
    {"metric": "Flooded road count", "total": row.sum_flooded_road_count, "mean": row.mean_flooded_road_count, "median": row.median_flooded_road_count, "min": row.min_flooded_road_count, "max": row.max_flooded_road_count},
    {"metric": "Flooded road miles", "total": row.sum_flooded_road_miles, "mean": row.mean_flooded_road_miles, "median": row.median_flooded_road_miles, "min": row.min_flooded_road_miles, "max": row.max_flooded_road_miles},
    {"metric": "Estimated damage", "total": row.sum_estimated_damage_cost, "mean": row.mean_estimated_damage_cost, "median": row.median_estimated_damage_cost, "min": row.min_estimated_damage_cost, "max": row.max_estimated_damage_cost},
    {"metric": "Exposed property value", "total": row.sum_estimated_exposed_property_value, "mean": row.mean_estimated_exposed_property_value, "median": row.median_estimated_exposed_property_value, "min": row.min_estimated_exposed_property_value, "max": row.max_estimated_exposed_property_value},
]
aggregate = pd.DataFrame(aggregate_rows)
for column in ["total", "mean", "median", "min", "max"]:
    aggregate[column] = aggregate[column].astype(object)
currency_metrics = {"Estimated damage", "Exposed property value"}
for idx, item in aggregate.iterrows():
    if item["metric"] in currency_metrics:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = money(float(item[column]))
    else:
        for column in ["total", "mean", "median", "min", "max"]:
            aggregate.loc[idx, column] = round(float(item[column]), 2)
aggregate

## 6. Scenario Trend Tables

In [ ]:
damage_chart

In [ ]:
roads_chart

In [ ]:
buildings_chart

In [ ]:
property_chart

## 7. Top Impacted Roads

In [ ]:
top_roads_plus_3ft = top_roads[top_roads["sea_level_rise_ft"] == 3.0].copy()
top_roads_plus_3ft["flooded_length_mi"] = top_roads_plus_3ft["flooded_length_mi"].round(3)
top_roads_plus_3ft[["study_area_name", "rank", "road_name", "mtfcc", "flooded_length_mi", "max_depth_ft"]].head(30)

## 8. Report Files

In [ ]:
for path in sorted(REPORT_DIR.glob("regional*.csv")):
    print(path.relative_to(PROJECT_ROOT))

## Key Caveats

- Norfolk uses the high-resolution 1-meter DEM workflow; the other six cities use the coarse regional DEM workflow.
- Results are screening-level static connected-inundation outputs, not a hydrodynamic simulation.
- Damage estimates use a simple depth-based replacement-cost model.
- Property-value exposure uses city-level ACS median owner-occupied home value as a uniform proxy, not parcel assessment values.
- Regional datum conversion currently uses Sewells Point; production work should evaluate spatially varying tidal datums or VDatum.